# 00 — Setup del entorno
**Spec:** docs/specs/00_entorno.md · **Requisitos:** R-37, R-38 · **Crea:** schemas `iol_bronze`, `iol_silver`, `iol_gold`, `iol_ops` y Volume `iol_bronze.landing`

Idempotente (`CREATE ... IF NOT EXISTS`): se puede correr las veces que haga falta.
El CSV original se sube a mano a `landing/trades/` (no está en el repo).

In [0]:
dbutils.widgets.text("pipeline_run_id", "")
dbutils.widgets.text("catalog", "workspace")

import json
import os
import platform
import sys
import uuid

sys.path.insert(0, os.path.abspath("../src"))
from iol_pipeline import config, io, smoke

RUN_ID = dbutils.widgets.get("pipeline_run_id") or str(uuid.uuid4())
CATALOG = dbutils.widgets.get("catalog")
print("pipeline_run_id:", RUN_ID)

pipeline_run_id: e3453c65-e571-4b25-98af-28e8127f3593


## Runtime

In [0]:
runtime = {
    "databricks_runtime_version": os.environ.get("DATABRICKS_RUNTIME_VERSION", "<no definido>"),
    "spark_version": spark.version,
    "python_version": platform.python_version(),
    "catalog": CATALOG,
}
for key, value in runtime.items():
    print(f"{key:28} {value}")

databricks_runtime_version   client.6.5
spark_version                4.2.0
python_version               3.12.3
catalog                      workspace


## Schemas y Volume

In [0]:
created = io.ensure_schemas(spark, catalog=CATALOG)
print("Asegurados:", created)
display(spark.sql(f"SHOW SCHEMAS IN {CATALOG} LIKE 'iol_*'"))
display(spark.sql(f"SHOW VOLUMES IN {CATALOG}.{config.SCHEMAS['bronze']}"))

Asegurados: ['workspace.iol_bronze', 'workspace.iol_silver', 'workspace.iol_gold', 'workspace.iol_ops', 'workspace.iol_bronze.landing']


databaseName
iol_bronze
iol_gold
iol_ops
iol_silver


database,volume_name
iol_bronze,landing


## Landing: subcarpetas y CSV de operaciones

In [0]:
for subdir in config.LANDING_SUBDIRS:
    dbutils.fs.mkdirs(io.landing_path(subdir, catalog=CATALOG))

trades_dir = io.landing_path("trades", catalog=CATALOG)
csv_files = [f for f in dbutils.fs.ls(trades_dir) if f.name.lower().endswith(".csv")]

if not csv_files:
    print(f"CSV_MISSING: subir el CSV original a {trades_dir}/ (Catalog Explorer > Volumes > Upload) y volver a correr.")
contract_ok = {}
for f in csv_files:
    columns = spark.read.option("header", True).csv(f.path).columns
    contract_ok[f.name] = columns == config.TRADES_CONTRACT
    print(f"{f.name}: {f.size:,} bytes · columnas = contrato {config.SCHEMA_VERSION}: {contract_ok[f.name]}")
    if not contract_ok[f.name]:
        print("  columnas leídas:", columns)

status = smoke.setup_status(contract_ok)
print("status:", status)

data-set-challenge-5-.csv: 12,242,003 bytes · columnas = contrato trades_v1: True
status: SUCCESS


In [0]:
dbutils.notebook.exit(json.dumps({
    "status": status,
    "pipeline_run_id": RUN_ID,
    "schemas": created,
    "csv_contract_ok": contract_ok,
}))